# V Group AI Assistant - prototype & test notebook (Phase 1 + Phase 2)

This notebook exercises the backend in `app/` end to end and ships a **Gradio** chat prototype.

**Phase 1 - grounded RAG**
1. Inspect the supplied V Group data (`Database/final_dataset.json`)
2. Ingestion: cleaning, metadata, FAQ extraction, chunking
3. Vector index + hybrid retrieval (configurable top-k / threshold)
4. Grounded RAG answers with sources, no-answer team hand-off, off-topic handling
5. Evaluation on the eval set built from the data

**Phase 2 - conversation**
6. Sessions & memory, follow-up questions
7. Service recommendations, sentiment-aware replies
8. Contact form flow, 3-attempt irrelevant-input rule, idle sessions
9. Transcript, emails (outbox) and feedback
10. FastAPI tests (in-process) and the Gradio chat app with notification sounds

**LLM:** set one of `ANTHROPIC_API_KEY`, `GROQ_API_KEY`, `OPENAI_API_KEY`, `GEMINI_API_KEY` in `.env` (or the cell below)
for answers written by the model. Without a key the pipeline runs in **offline mode** (answers are built from sentences
taken verbatim from the retrieved pages). Emails are written to `storage/outbox/*.eml` unless SMTP is configured.

In [ ]:
# Install dependencies if needed (uncomment):
# %pip install -r ../requirements.txt

In [ ]:
import os, sys, json, collections
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "0")   # set to "1" once the embedding model is cached

# Optional but recommended: an LLM key gives fully conversational answers written in its own words
# (still grounded in the retrieved V Group pages, with citations). Without a key, answers are built
# from cleaned-up sentences taken verbatim from the pages. Never hard-code keys in the notebook.
# import getpass; os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
# import getpass; os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

from app.config import Settings
from app.logging_config import setup_logging
setup_logging("WARNING")
settings = Settings()
print("data:", settings.data_path)
print("vector store:", settings.vector_store, "| embedding:", settings.embedding_model)
print("top_k:", settings.top_k, "| similarity_threshold:", settings.similarity_threshold,
      "| domain_threshold:", settings.domain_threshold)

## 1. Inspect the supplied data

In [ ]:
from app.ingestion.loader import load_raw
raw = load_raw(settings.data_path)
print("records:", len(raw))
print("fields:", collections.Counter(k for r in raw for k in r))
print("source_type:", collections.Counter(r["source_type"] for r in raw))
print("platform:", collections.Counter(r["platform"] or "(empty)" for r in raw))
lens = sorted(len(r["text"]) for r in raw)
print("text length min/median/max:", lens[0], lens[len(lens)//2], lens[-1])
print("records with < 40 chars of text:", sum(l < 40 for l in lens))

## 2. Ingestion: cleaning, metadata, FAQs, chunking

In [ ]:
from app.ingestion.loader import load_documents
from app.ingestion.chunker import chunk_documents

docs = load_documents(settings.data_path)
print("documents after cleaning:", len(docs))
print("source_type:", collections.Counter(d.source_type for d in docs))
print("category:", collections.Counter(d.category for d in docs))
print("content_quality:", collections.Counter(d.content_quality for d in docs))
print("FAQ pairs extracted:", sum(len(d.faqs) for d in docs))

d = next(d for d in docs if d.page_id == "db-425")
print("\nExample document:", d.title, "|", d.category, "|", d.url)
print("keywords:", d.keywords, "| CTAs:", d.ctas)
for f in d.faqs[:3]:
    print("  Q:", f.question, "\n  A:", f.answer)

In [ ]:
chunks = chunk_documents(docs, settings.chunk_size, settings.chunk_overlap, settings.min_chunk_chars)
sizes = [len(c.text) for c in chunks]
print("chunks:", len(chunks), "| page chunks:", sum(c.content_type == "page" for c in chunks),
      "| faq chunks:", sum(c.content_type == "faq" for c in chunks))
print("chunk chars min/avg/max:", min(sizes), sum(sizes)//len(sizes), max(sizes))
print("metadata example:", json.dumps(chunks[0].metadata, indent=1)[:600])

## 3. Build / load the index and test retrieval
First run embeds ~950 chunks with `BAAI/bge-base-en-v1.5` (a few minutes on CPU). Later runs reuse the
persisted index unless the data or chunk settings change.

In [ ]:
from app.rag.pipeline import build_pipeline
bundle = build_pipeline(settings)
pipeline, retriever = bundle.pipeline, bundle.pipeline.retriever
print("index:", bundle.index_report)
print("answer mode:", pipeline.mode)

In [ ]:
def show_retrieval(q, top_k=5, threshold=None):
    res = retriever.retrieve(q, top_k=top_k, similarity_threshold=threshold)
    print(f"Q: {q}  (top score {res.top_score:.3f}, {len(res.hits)} hits above threshold)")
    for h in res.hits:
        m = h.metadata
        print(f"  {h.score:.3f} (dense {m['dense_score']:.3f}, lexical {m['lexical_score']:.2f}) "
              f"[{m['content_type']}] {m['title']} - {m['url']}")

for q in ["How many support hours are included annually?", "What's your phone number?",
          "What is the price of the BigCommerce Gold plan?", "Tell me a joke"]:
    show_retrieval(q)
    print()

## 4. RAG answers: grounded answers + sources, team hand-off, off-topic

In [ ]:
def ask(q, **kw):
    r = pipeline.answer(q, **kw)
    print(f"Q: {q}\n[{r.status} | {r.request_kind} | top {r.top_score:.2f} | {r.generator} | {r.latency_ms} ms]")
    print("A:", r.answer)
    for s in r.sources:
        print(f"   [{s.number}] {s.title} ({s.source_type}) {s.url}")
    if r.action:
        print("   -> UI action:", r.action["type"], [f["name"] for f in r.action["fields"]])
    print()

for q in ["what do you guys do??",
          "What's your phone number?",
          "What is included in V Group's Annual Support?",
          "How much does the Shopify Gold plan cost?",
          "What is ReviewCaddy?",
          "Can you migrate my X-Cart store to WooCommerce?",
          "I want a blockchain-based NFT marketplace",          # in-domain, no answer -> great idea + form
          "What is your refund policy?",                       # info not in data -> team + form
          "What is the weather in Paris today?",               # off-topic
          "hi"]:
    ask(q)

## 5. Evaluation on the eval set built from the supplied data

In [ ]:
from app.evaluation import load_eval_set, evaluate_retrieval, evaluate_pipeline
eval_set = load_eval_set()
ret = evaluate_retrieval(retriever, eval_set["answerable"], k=5)
print("retrieval:", {k: v for k, v in ret.items() if k != "failures"}, "failures:", ret["failures"])
e2e = evaluate_pipeline(pipeline, eval_set)
print(json.dumps({k: v for k, v in e2e.items() if k != "misses"}, indent=2))
print("misses:", json.dumps(e2e["misses"], indent=1))

## 6. Phase 2 services: sessions and memory
`build_services` wires the conversation manager, session store, recommendations, leads (SQLite), emails and
feedback around the pipeline built above. The demo below uses a **simulated clock** so idle timeouts can be shown
instantly, and an in-memory mailbox so emails can be printed.

In [ ]:
from datetime import datetime, timedelta, timezone
from app.services import build_services
from app.email.senders import MemoryEmailSender
from app.leads.service import LeadInput

class DemoClock:
    def __init__(self): self.now = datetime.now(timezone.utc)
    def __call__(self): return self.now
    def advance(self, seconds): self.now += timedelta(seconds=seconds)

demo_clock, mailbox = DemoClock(), MemoryEmailSender()
demo_settings = settings.model_copy(update={"storage_dir": settings.storage_dir / "notebook_demo"})
svc = build_services(demo_settings, bundle, clock=demo_clock, email_sender=mailbox, background_email=False)
manager = svc.manager
print("catalog items for recommendations:", len(svc.catalog.items))

def say(sid, text):
    demo_clock.advance(10)
    r = manager.handle_message(sid, text)
    print(f"YOU: {text}")
    print(f"BOT [{r.status} | sentiment: {r.sentiment.label} | attempts left: {r.attempts_remaining}]:")
    print("   " + r.reply.content.replace("\n", "\n   "))
    if r.rag and r.rag.search_query != text:
        print("   (searched as:", r.rag.search_query, ")")
    if r.recommendations:
        rec = r.recommendations
        print("   RECOMMENDED:", rec["primary"]["name"], "| related:", [x["name"] for x in rec["related"]])
    if r.action:
        print("   UI ACTION:", r.action["type"])
    print()
    return r

In [ ]:
# Multi-turn memory + follow-up questions
sid = manager.start_session().session.session_id
say(sid, "How much does the Shopify Gold plan cost?")
say(sid, "And how long does it take to deliver?")      # 'it' -> Shopify Packages
say(sid, "What about BigCommerce?")                    # ellipsis -> same question for BigCommerce
say(sid, "Tell me about ReviewCaddy")
say(sid, "Does it work with Shopify?")
print("stored messages:", len(manager.get(sid).messages))

## 7. Recommendations and sentiment-aware replies

In [ ]:
sid = manager.start_session().session.session_id
say(sid, "I need a mobile app for my business, what do you recommend?")
say(sid, "I'm having a problem - how many hours does Annual Support include?")   # negative -> brief acknowledgement
say(sid, "This is useless, you are not helping at all!!")                      # frustrated -> offer the team

## 8. Contact form flow, irrelevant-input limit, idle sessions

In [ ]:
sid = manager.start_session().session.session_id
say(sid, "I want to talk to your sales team")
r = manager.submit_contact(sid, LeadInput(name="Demo Customer", email="demo.customer@example.com",
                                          message="Interested in a Shopify Gold package"))
print("FORM SUBMITTED ->", r.reply.content, "| interest recorded:", r.lead.interest, "\n")
say(sid, "No thanks")      # -> polite close + feedback prompt + summary email
print("closed because:", manager.get(sid).close_reason)
print("emails sent:", [(e.email_type, e.to, e.subject) for e in mailbox.sent])

In [ ]:
# 3-attempt rule (configurable with IRRELEVANT_LIMIT)
sid = manager.start_session().session.session_id
for q in ["What is the weather in Paris today?", "Tell me a joke", "Who won the 2022 world cup?"]:
    say(sid, q)

In [ ]:
# Idle session: 'are you still there?' -> wait -> close + follow-up email (timings from settings)
sid = manager.start_session(email="idle.demo@example.com").session.session_id
say(sid, "What is SMTU?")
session = manager.get(sid)
demo_clock.advance(demo_settings.idle_check_after_seconds)
print("after", demo_settings.idle_check_after_seconds, "s idle ->", [m.content for m in manager.check_idle(session)])
demo_clock.advance(demo_settings.idle_close_after_seconds)
print("after another", demo_settings.idle_close_after_seconds, "s ->", [m.content for m in manager.check_idle(session)])
print("follow-up email:", mailbox.sent[-1].subject, "| attachment:", [a.filename for a in mailbox.sent[-1].attachments])

## 9. Transcript, emails and feedback

In [ ]:
from app.transcripts.builder import build_transcript
print(build_transcript(manager.get(sid)))

In [ ]:
email_msg = mailbox.sent[-1]
print("SUBJECT:", email_msg.subject, "\nTO:", email_msg.to, "\n")
print(email_msg.text)

In [ ]:
fb_session = manager.start_session().session.session_id
print(manager.record_feedback(fb_session, "Great").content)
print("stored:", svc.feedback.get(fb_session))

## 10. FastAPI endpoint tests (in-process, no server needed)

In [ ]:
from fastapi.testclient import TestClient
from app.main import create_app

api = create_app()
api.state.services = svc             # reuse the services built above
api.state.owns_services = False
with TestClient(api) as client:
    print(client.get("/health").json())
    print("ask:", client.post("/ask", json={"question": "Do you offer Drupal development services?"}).json()["answer"][:150])
    sid = client.post("/sessions", json={"name": "API Demo"}).json()["session"]["session_id"]
    r = client.post(f"/sessions/{sid}/messages", json={"message": "How much does the Shopify Gold plan cost?"}).json()
    print("chat:", r["status"], "|", r["message"]["content"][:120], "| sound:", r["notification"]["sound"])
    r = client.post(f"/sessions/{sid}/messages", json={"message": "And how long does it take?"}).json()
    print("follow-up:", r["message"]["content"][:120])
    print("transcript:", client.get(f"/sessions/{sid}/transcript").headers["content-disposition"])
    print("bad message:", client.post(f"/sessions/{sid}/messages", json={"message": ""}).status_code)
    print("unknown session:", client.get("/sessions/nope").status_code)
    print("bad contact:", client.post(f"/sessions/{sid}/contact", json={"name": "A", "email": "x", "message": "hi"}).status_code)

## 11. Gradio chat prototype (Phase 2)
Real-time clock, demo idle timings, emails written to `storage/outbox`. Features: chat with memory, sources and
recommendations panel, contact form, feedback buttons, transcript download/email, **notification sounds** with an
on/off toggle (sent / received / idle check / chat ended).

In [ ]:
from app.ui.gradio_app import build_demo

ui_settings = settings.model_copy(update={
    "idle_check_after_seconds": 60,     # shortened for the demo; production default is 120
    "idle_close_after_seconds": 30,
    "email_backend": "file",            # emails -> storage/outbox/*.eml
})
ui_services = build_services(ui_settings, bundle)
demo, head = build_demo(ui_services)
demo.launch(head=head, inbrowser=False)

In [ ]:
# demo.close()